**O3 模型・重點小結**

1. **核心問題**

   * 你要分析一批文本，想先用 **GAP**（Generalized Association Plots）做可視化分群，再用 **LDA**（Latent Dirichlet Allocation）或其監督式變體（如 Labeled LDA）來取得每篇文本的主題分佈，判斷「偏向哪個主題」。

2. **兩種流程比較**

   | 流程 | 先做 GAP → 再做 LDA         | 先指定 topic→token → GAP 分群 → LDA 驗證 |
   | -- | ----------------------- | --------------------------------- |
   | 用途 | 讓可視化主動揭示隱含群，再以 LDA 量化主題 | 用專家/先驗定義 token 群，再檢驗實際分布          |
   | 優點 | 完全資料驅動，能發現意外結構          | 可驗證人為標籤、易對照 domain 知識             |
   | 風險 | GAP 聚類不一定對應可解釋主題        | 預先 token list 可能偏誤，限制 LDA 彈性      |

3. **主題歸屬 → 延伸應用**

   * **推薦系統**：把「文件–主題分布」對應成「使用者/商品–潛在偏好」，即可做內容推薦或商品推薦。
   * **顧客分析**：將每位顧客的行為紀錄視為文件，LDA 得到「顧客–主題偏好向量」，GAP 幫你做群組視覺化，便於客群分層、行銷定位。

4. **實作建議**

   1. **訓練**：對已標或未標文本跑 LDA（或 Labeled LDA 如果有多重標籤）。
   2. **推論**：輸入新文本，推斷其 `θ`（topic mixture），即得到「偏向哪個主題」的機率。
   3. **可視化／分群**：用 GAP 依 `θ` 或相似度排序，觀察群落與主題塊狀。
   4. **應用**：

      * 推薦：以使用者對主題的偏好去匹配文件或商品。
      * 分群：以顧客在主題空間的位置做 segmentation。

5. **下一步**

   * 若想快速驗證，可先用現成 LDA 套件（Gensim／sklearn）＋ R 版 GAP。
   * 需要設計推薦或顧客分析流程，再進一步整合「主題 → 相似度 → 排薦」的演算法。


In [9]:
import pandas as pd
import re, string

# 下載並設定停用詞
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = stopwords.words('english')

from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
data = pd.read_csv(r'C:\Users\USER\Documents\GitHub\psychic-spoon\BERTopic\bbc-news-data.csv', sep='\t')
data.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


business:
  - market, share, profit, loss, company, merger, revenue, investor, bank, finance, economy, bn, year, sales, growth

entertainment:
  - film, movie, music, album, actor, actress, show, theatre, festival, award, star

politics:
  - government, minister, election, parliament, policy, vote, party, labour, conservative, law

sport:
  - match, win, defeat, league, cup, coach, player, score, goal, season

tech:
  - software, internet, computer, mobile, digital, iphone, microsoft, google, technology, device

  Top words for business: ['bn', 'mr', 'year', 'sales', 'growth', 'market', 'economy', 'company', 'bank', 'new', 'oil', 'firm', 'economic', 'shares', 'government']
Top words for entertainment: ['film', 'said', 'best', 'music', 'number', 'new', 'years', 'year', 'award', 'awards', 'band', 'mr', 'album', 'films', 'festival']
Top words for politics: ['mr', 'labour', 'election', 'blair', 'party', 'government', 'people', 'brown', 'howard', 'minister', 'tax', 'new', 'lord', 'prime', 'told']
Top words for sport: ['said', 'game', 'england', 'world', 'win', 'play', 'players', 'cup', 'wales', 'time', 'match', 'club', 'good', 'team', 'chelsea']
Top words for tech: ['people', 'mobile', 'games', 'mr', 'music', 'technology', 'software', 'new', 'users', 'game', 'phone', 'digital', 'net', 'broadband', 'computer']

In [10]:
def preprocess(text):
    text = text.lower()  # 轉成小寫
    text = re.sub(r'\d+', '', text)  # 移除數字
    text = re.sub(r'[^\w\s]', '', text)  # 移除標點符號
    text = re.sub(r'\s+', ' ', text).strip()  # 移除多餘空白
    text = ' '.join([word for word in text.split() if word not in stop_words])  # 移除停用詞
    return text

data['cleaned_content'] = data['content'].apply(preprocess)
data['Tokens'] = data['cleaned_content'].apply(lambda x: x.split())

---

### 直接設定seed

In [ ]:
seed = {
    'business':['market', 'share', 'profit', 'loss', 'company', 'merger', 'revenue', 'investor', 'bank', 'finance', 'economy'],
    'entertaiment':['film', 'movie', 'music', 'album', 'actor', 'actress', 'show', 'theatre', 'festival', 'award', 'star'],
    'politics':['government', 'minister', 'election', 'parliament', 'policy', 'vote', 'party', 'labour', 'conservative', 'law'],
    'sport':['match', 'win', 'defeat', 'league', "cup", 'coach', 'player', 'score', 'goal', 'season'],
    'tech':['software', 'internet', 'computer', 'mobile', 'digital', 'iphone', 'microsoft', 'google', 'technology', 'device']
}

topic_cols = list(seed.keys())
scores = []

for doc in data['cleaned_content']:
    wc = Counter(doc.split())
    vec = [sum(wc[token] for token in seed[k]) for k in topic_cols]
    scores.append(vec)

bbcnews_topic = pd.DataFrame(scores, columns=topic_cols)
data = pd.concat([data, bbcnews_topic], axis=1)
data['Tokens'] = data['cleaned_content'].apply(lambda x: x.split())

data.to_csv('bbcnews_topic_matrix.csv', index=False)

# df_subset = data[['category', 'business', 'entertaiment', 'politics', 'sport', 'tech']]
# df_subset.to_csv('bbc_seed_for_gap1.csv', index=False)


---

### 算文章的tfidf來設定seed

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
for label in data['category'].unique():
    docs = data[data['category'] == label]['cleaned_content']
    vec = TfidfVectorizer(max_df=0.8, min_df=10, stop_words='english')
    tfidf = vec.fit_transform(docs)
    top_words = sorted(zip(vec.get_feature_names_out(), 
                           tfidf.sum(axis=0).A1),
                           key=lambda x: -x[1])[:15]
    print(f'Top words for {label}: {[w for w, _ in top_words]}')

Top words for business: ['bn', 'mr', 'year', 'sales', 'market', 'growth', 'company', 'economy', 'new', 'bank', 'firm', 'oil', 'shares', 'economic', 'government']
Top words for entertainment: ['film', 'said', 'best', 'music', 'new', 'years', 'number', 'year', 'awards', 'award', 'band', 'films', 'mr', 'album', 'festival']
Top words for politics: ['mr', 'labour', 'election', 'blair', 'party', 'government', 'people', 'brown', 'minister', 'new', 'howard', 'lord', 'tax', 'told', 'prime']
Top words for sport: ['said', 'game', 'england', 'world', 'win', 'play', 'players', 'cup', 'wales', 'time', 'match', 'club', 'good', 'team', 'open']
Top words for tech: ['people', 'mobile', 'games', 'mr', 'software', 'technology', 'music', 'game', 'new', 'users', 'phone', 'digital', 'net', 'computer', 'use']


In [12]:
seed = {
    'business':['bn', 'mr', 'year', 'sales', 'market', 'growth', 'company', 'economy', 'new', 'bank', 'firm', 'oil', 'shares', 'economic', 'government'],
    'entertaiment':['film', 'said', 'best', 'music', 'new', 'years', 'number', 'year', 'awards', 'award', 'band', 'films', 'mr', 'album', 'festival'],
    'politics':['mr', 'labour', 'election', 'blair', 'party', 'government', 'people', 'brown', 'minister', 'new', 'howard', 'lord', 'tax', 'told', 'prime'],
    'sport':['said', 'game', 'england', 'world', 'win', 'play', 'players', 'cup', 'wales', 'time', 'match', 'club', 'good', 'team', 'open'],
    'tech':['people', 'mobile', 'games', 'mr', 'software', 'technology', 'music', 'game', 'new', 'users', 'phone', 'digital', 'net', 'computer', 'use']
}

selected_tokens = set()
for words in seed.values():
    selected_tokens.update(words)

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(vocabulary=list(selected_tokens))
X = vectorizer.fit_transform(data['cleaned_content'])

tf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tf_df.to_csv("filtered_doc_token_matrix2.csv", index=False)


---

### 建立 token 白名單

In [ ]:
seed = {
    'business':['market', 'share', 'profit', 'loss', 'company', 'merger', 'revenue', 'investor', 'bank', 'finance', 'economy'],
    'entertaiment':['film', 'movie', 'music', 'album', 'actor', 'actress', 'show', 'theatre', 'festival', 'award', 'star'],
    'politics':['government', 'minister', 'election', 'parliament', 'policy', 'vote', 'party', 'labour', 'conservative', 'law'],
    'sport':['match', 'win', 'defeat', 'league', "cup", 'coach', 'player', 'score', 'goal', 'season'],
    'tech':['software', 'internet', 'computer', 'mobile', 'digital', 'iphone', 'microsoft', 'google', 'technology', 'device']
}

selected_tokens = set()
for words in seed.values():
    selected_tokens.update(words)

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(vocabulary=list(selected_tokens))
X = vectorizer.fit_transform(data['cleaned_content'])

tf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tf_df.to_csv("filtered_doc_token_matrix.csv", index=False)

---

In [13]:
from gensim.corpora import Dictionary
dictionary = Dictionary(data['Tokens'])
corpus = [dictionary.doc2bow(tokens) for tokens in data['Tokens']]

selftoken_df = pd.read_csv('r_token_to_topic2.csv')
token_to_topic = dict(zip(selftoken_df['token'], selftoken_df['topic']))

In [14]:
import numpy as np
K = 5
V = len(dictionary.token2id)
eta = np.full((K, V), 0.01)

for word, topic_id in token_to_topic.items():
    if topic_id < K and word in dictionary.token2id:
        word_id = dictionary.token2id[word]
        eta[topic_id][word_id] = 0.01

In [15]:
from gensim.models.ldamodel import LdaModel

lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=K,
               eta=eta, random_state=42, iterations=100, passes=10)


In [16]:
for k in range(K):
    top_words = lda.show_topic(k, topn=15)
    print(f"Topic {k}: {[w for w, _ in top_words]}")

Topic 0: ['music', 'people', 'mobile', 'said', 'games', 'digital', 'game', 'also', 'technology', 'players', 'one', 'would', 'gaming', 'time', 'new']
Topic 1: ['said', 'mr', 'new', 'music', 'games', 'technology', 'mac', 'apple', 'computer', 'pc', 'also', 'would', 'dvd', 'sales', 'mini']
Topic 2: ['said', 'mr', 'would', 'us', 'also', 'year', 'last', 'one', 'new', 'years', 'first', 'government', 'two', 'bn', 'best']
Topic 3: ['said', 'people', 'users', 'software', 'phone', 'net', 'one', 'security', 'search', 'technology', 'broadband', 'use', 'many', 'data', 'also']
Topic 4: ['said', 'people', 'could', 'tv', 'services', 'new', 'digital', 'would', 'says', 'make', 'mr', 'european', 'service', 'one', 'technology']


In [17]:
for i, bow in enumerate(corpus[:10]):
    topic_dist = lda.get_document_topics(bow)
    print(f"Doc {i} → {topic_dist}")


Doc 0 → [(1, 0.13489039), (2, 0.53495735), (3, 0.2932971), (4, 0.035934564)]
Doc 1 → [(0, 0.012494434), (1, 0.028866438), (2, 0.8625727), (4, 0.09511209)]
Doc 2 → [(1, 0.0578888), (2, 0.9120359), (4, 0.027255839)]
Doc 3 → [(0, 0.11654742), (1, 0.20938951), (2, 0.6387682), (3, 0.03436851)]
Doc 4 → [(2, 0.62843263), (3, 0.03992384), (4, 0.32869393)]
Doc 5 → [(2, 0.7663248), (3, 0.128735), (4, 0.10072887)]
Doc 6 → [(2, 0.741504), (4, 0.25466207)]
Doc 7 → [(2, 0.85722375), (3, 0.018500978), (4, 0.12204012)]
Doc 8 → [(0, 0.012342449), (1, 0.20877628), (2, 0.6995857), (4, 0.07760019)]
Doc 9 → [(1, 0.13566358), (2, 0.85956943)]
